In [ ]:
from pyspark.sql import SparkSession

In [ ]:
# สร้าง Spark Session
spark = SparkSession.builder \
    .appName("Get Data") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 09:29:27 INFO SparkEnv: Registering MapOutputTracker
26/04/26 09:29:27 INFO SparkEnv: Registering BlockManagerMaster
26/04/26 09:29:27 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/26 09:29:27 INFO SparkEnv: Registering OutputCommitCoordinator


# Clean Post Data

In [ ]:
col_post_selected = ['content_categories',
                     'created_utc',
                     'author',
                     'is_crosspostable',
                     'num_comments',
                     'num_crossposts',
                     'selftext',
                     'subreddit',
                     'title',
                     'upvote_ratio',
                     'ups',
                     'downs',
                     'view_count']

paths = ["gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_Anthropic/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_Bard/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_ChatGPT/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_ChatGPTPro/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_ClaudeAI/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_DeepSeek/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_GeminiAI/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_OpenAI/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_grok/part*"]


post = spark.read.parquet(*paths).select(col_post_selected)

In [ ]:
post.printSchema()
post.cache()

root
 |-- content_categories: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- author: string (nullable = true)
 |-- is_crosspostable: boolean (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- num_crossposts: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- title: string (nullable = true)
 |-- upvote_ratio: double (nullable = true)
 |-- ups: long (nullable = true)
 |-- downs: long (nullable = true)
 |-- view_count: string (nullable = true)



DataFrame[content_categories: string, created_utc: bigint, author: string, is_crosspostable: boolean, num_comments: bigint, num_crossposts: bigint, selftext: string, subreddit: string, title: string, upvote_ratio: double, ups: bigint, downs: bigint, view_count: string]

In [ ]:
print(f"Number of rows is: {post.count()}")

Number of rows is: 428771


In [ ]:
post = post.dropna(how='any', subset = 'title')

print(f'After Drop missing value from title: {post.count()}')

After Drop missing value from title: 428771


In [ ]:
from pyspark.sql.functions import length, percentile_approx
from pyspark.sql import functions as F

# Calculate the length of the 'selftext' column, treating nulls as empty strings
post_with_text_length = post.withColumn("char_count", F.length(post['title']))

# Calculate the 75th percentile of 'selftext_length' ยอมให้คลาดเคลื่อนได้ 5% แลกกับความเร็วและประหยัด Cost
p75 = post_with_text_length.approxQuantile("char_count", [0.75], 0.05)[0]

# Drop rows where 'selftext_length' is greater than the 75th percentile
post = post_with_text_length.filter(F.col("char_count") <= p75).drop("char_count")

print(f"75th percentile of selftext length: {p75}")
print(f"Number of rows after filtering: {post.count()}")

75th percentile of selftext length: 67.0
Number of rows after filtering: 314849


## Convert time in epoch format to Date Time

In [ ]:
# Or using from_unixtime for a specific string format
post = post.withColumn("created_utc", F.from_unixtime(F.col("created_utc"), "yyyy-MM-dd HH:mm:ss"))
print("Convert Epoch to Data Time Task is Complete")

Convert Epoch to Data Time Task is Complete


## Create Column for Taging AI Service

In [ ]:
# Convert all title to lower case for appropiate matching with nest step
post = post.withColumn("title", F.lower(F.col("title")))
print("Convert Title to Lower Case")

Convert Title to Lower Case


In [ ]:
# Create ChatGPT_Tag
post = post.withColumn("ChatGPT_Tag",
                        F.when(
                            (F.col("title").contains("chatgpt")) |
                            (F.col("title").contains("openai")) |
                            (F.col("title").contains(" gpt")) | # เพิ่ม space ข้างหน้าเพื่อเลี่ยงคำอื่นที่สะกดคล้ายกัน
                            (F.col("title").contains("dall-e")) | # เพิ่ม DALL-E เพราะเป็น Product ของ OpenAI
                            (F.col("title").contains("sora")),    # เพิ่ม Sora
                            1
                              ).otherwise(0)
                      )

print("Create ChatGPT_Tag Complete")

# Create ClaudeAI_Tag
post = post.withColumn("ClaudeAI_Tag",
                        F.when(
                            (F.col("title").contains("claude")) |
                            (F.col("title").contains("anthropic")),
                            1
                        ).otherwise(0)
                      )

print("Create ClaudeAI_Tag Complete")

# Create Gemini_Tag
post = post.withColumn("GeminiAI_Tag",
                        F.when(
                            (F.col("title").contains("gemini")) |
                            (F.col("title").contains("bard")) |
                            (F.col("title").contains("google ai")) |
                            (F.col("title").contains("palm")), # Model พื้นฐานของ Google
                            1
                              ).otherwise(0)
                      )

print("Create GeminiAI_Tag Complete")

# Create DeepSeek_Tag
post = post.withColumn("DeepSeek_Tag",
                        F.when(
                            (F.col("title").contains("deepseek")) |
                            (F.col("title").contains("deep seek")), # บางคนพิมพ์แยกคำ
                            1
                              ).otherwise(0)
                      )

print("Create DeepSeek_Tag Complete")

# Create Grok_Tag
post = post.withColumn("Grok_Tag",
                        F.when(
                            (F.col("title").contains("grok")) |
                            (F.col("title").contains(" xai")), # ชื่อบริษัทของ Elon Musk ที่ทำ Grok
                            1
                        ).otherwise(0)
                      )

print("Create Grok_Tag Complete")

Create ChatGPT_Tag Complete
Create ClaudeAI_Tag Complete
Create GeminiAI_Tag Complete
Create DeepSeek_Tag Complete
Create Grok_Tag Complete


## Split Post by Tag Count

In [ ]:
# Create Tag_Count

post = post.withColumn("Tag_Count",
                       F.col("ChatGPT_Tag") +
                       F.col("ClaudeAI_Tag") +
                       F.col("GeminiAI_Tag") +
                       F.col("DeepSeek_Tag") +
                       F.col("Grok_Tag"))

print("Create Tag_Count Complete")

Create Tag_Count Complete


In [ ]:
post.groupBy("Tag_Count").count().sort("count",ascending=False).show()

+---------+------+
|Tag_Count| count|
+---------+------+
|        0|198429|
|        1|112325|
|        2|  3795|
|        3|   250|
|        4|    46|
|        5|     4|
+---------+------+



In [ ]:
# Split DataSet by Tag_Count
# Dataset where there is 0 or 1 AI mentioned
post_single_tag_or_none = post.filter(F.col("Tag_Count") <= 1)

print(f"Total Rows: {post.count()}")
print(f"Single/None Tag Rows: {post_single_tag_or_none.count()}")

post_single_tag_or_none.cache()

# Dataset where multiple AI products are mentioned (Mixed content)
post_multi_tag = post.filter(F.col("Tag_Count") > 1)

print(f"Multi-Tag Rows: {post_multi_tag.count()}")

Total Rows: 314849


Single/None Tag Rows: 310754


Multi-Tag Rows: 4095


In [ ]:
post.unpersist()

DataFrame[content_categories: string, created_utc: string, author: string, is_crosspostable: boolean, num_comments: bigint, num_crossposts: bigint, selftext: string, subreddit: string, title: string, upvote_ratio: double, ups: bigint, downs: bigint, view_count: string, ChatGPT_Tag: int, ClaudeAI_Tag: int, GeminiAI_Tag: int, DeepSeek_Tag: int, Grok_Tag: int, Tag_Count: int]

In [ ]:
# Save post_multi_tag on data lake
target_path_parquet = "gs://reddit-ai-2/process_data/Post_Multi_Tag/post_multi_tag.parquet"

post_multi_tag.write.mode("overwrite").parquet(target_path_parquet)

print(f"บันทึกไฟล์ Parquet ไปที่ {target_path_parquet} เรียบร้อยแล้ว")

post_multi_tag.unpersist()

บันทึกไฟล์ Parquet ไปที่ gs://reddit-ai-2/process_data/Post_Multi_Tag/post_multi_tag.parquet เรียบร้อยแล้ว


DataFrame[content_categories: string, created_utc: string, author: string, is_crosspostable: boolean, num_comments: bigint, num_crossposts: bigint, selftext: string, subreddit: string, title: string, upvote_ratio: double, ups: bigint, downs: bigint, view_count: string, ChatGPT_Tag: int, ClaudeAI_Tag: int, GeminiAI_Tag: int, DeepSeek_Tag: int, Grok_Tag: int, Tag_Count: int]

## Transform Multi Tag to a categorical tag

In [ ]:
# Transform multi-columns for tagging to a categorical AI tag for ease to use.
from pyspark.sql.types import StringType

def subreddit_mapping(subreddit):
  if subreddit in ['ChatGPT','ChatGPTPro','OpenAI']:
    return "ChatGPT"
  elif subreddit in ['ClaudeAI','Anthropic']:
    return "Claude"
  elif subreddit in ['GeminiAI','Bard']:
    return "Gemini"
  elif subreddit in ['DeepSeek']:
    return "DeepSeek"
  elif subreddit in ['grok']:
    return "Grok"
  else:
    return "Cann't Identify"

def tag_mapping(tag1,tag2,tag3,tag4,tag5):
  if tag1 == 1:
    return "ChatGPT"
  elif tag2 == 1:
    return "Claude"
  elif tag3 == 1:
    return "Gemini"
  elif tag4 == 1:
    return "DeepSeek"
  elif tag5 == 1:
    return "Grok"
  else:
    return "Cann't Identify"

# Assign to Spark by UDF
subreddit_mapping_udf = F.udf(subreddit_mapping, StringType())
tag_mapping_udf = F.udf(tag_mapping, StringType())

In [ ]:
post_single_tag_or_none = post_single_tag_or_none.withColumn("AI_Name",
    F.when(
        F.col("Tag_Count") == 0,
        subreddit_mapping_udf(F.col("subreddit")) # Case 1: ใช้ค่าจาก column 'reddit' (สมมติว่าชื่อ 'subreddit')
    ).when(
        F.col("Tag_Count") == 1,
        tag_mapping_udf(
            F.col("ChatGPT_Tag"),
            F.col("ClaudeAI_Tag"),
            F.col("GeminiAI_Tag"),
            F.col("DeepSeek_Tag"),
            F.col("Grok_Tag")
        ) # Case 2: ใส่ Input 5 ตัวตาม Tag
    ).otherwise("Multi-Tag or Other") # กรณี Tag_Count > 1
)

print("Create AI Name Task is Complete")

Create AI Name Task is Complete


In [ ]:
post_single_tag_or_none.groupBy("AI_Name").count().show()

+--------+------+
| AI_Name| count|
+--------+------+
|DeepSeek|  9013|
| ChatGPT|191713|
|  Gemini| 36440|
|  Claude| 41765|
|    Grok| 31823|
+--------+------+



In [ ]:
# Delete Insufficient feature.
post_single_tag_or_none = post_single_tag_or_none.drop(
    "ChatGPT_Tag",
    "ClaudeAI_Tag",
    "GeminiAI_Tag",
    "DeepSeek_Tag",
    "Grok_Tag",
    "Tag_Count"
)

print("Delete Insufficient Feature")

Delete Insufficient Feature


## According to 'post_single_tag_or_none' Split to 2 dataset
1. Sampling 100,000 rows for taged sentiment by LLM and Train Model for predict sentiment for the rest post dataset
2. The Rest rows


In [ ]:
from pyspark.sql import functions as F

# 1. Add a unique ID to each row so we can track what was selected
post_single_tag_or_none = post_single_tag_or_none.withColumn("row_id", F.monotonically_increasing_id())

# 2. Calculate the fraction needed to get approximately 100,000 rows
total_count = post_single_tag_or_none.count()
fraction = 50000 / total_count

# 3. Create the sampling (Selected data)
# Note: sample() is probabilistic, so we use limit(100000) to get exactly that amount
sampling = post_single_tag_or_none.sample(withReplacement=False, fraction=fraction * 1.1, seed=42).limit(100000)

# 4. Store the rest of the data (Not selected)
# We use a Left Anti Join to find rows in post_with_id that are NOT in sampling
rest_data = post_single_tag_or_none.join(sampling, on="row_id", how="left_anti")

# Optional: Remove the row_id column if you don't need it anymore
sampling = sampling.drop("row_id")
rest_data = rest_data.drop("row_id")

# บน Dataproc ถ้าไม่ unpersist หน่วยความจำอาจจะเต็มสำหรับ Job ถัดไป
post_single_tag_or_none.unpersist()

print(f"Sampling count: {sampling.count()}")
print(f"Rest data count: {rest_data.count()}")

Sampling count: 55147
Rest data count: 255607


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit

# Define a window specification to generate a sequential number
window_spec = Window.orderBy(lit(1)) # Use a literal to create a single partition for ordering

# Add the new column with repeating sequence 1,2,3,4,5
sampling = sampling.withColumn("Subgroup", (row_number().over(window_spec) - 1) % 5 + 1)

sampling.show(5)

26/04/26 09:35:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:35:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:35:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:35:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:35:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:35:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 0

+------------------+-------------------+------------------+----------------+------------+--------------+--------------------+---------+--------------------+-------------------+---+-----+----------+-------+--------+
|content_categories|        created_utc|            author|is_crosspostable|num_comments|num_crossposts|            selftext|subreddit|               title|       upvote_ratio|ups|downs|view_count|AI_Name|Subgroup|
+------------------+-------------------+------------------+----------------+------------+--------------+--------------------+---------+--------------------+-------------------+---+-----+----------+-------+--------+
|              NULL|2025-01-01 02:14:30|        owningface|            true|           6|             0|                    |  ChatGPT|gpt says it can s...|0.28999999165534973|  0|    0|      NULL|ChatGPT|       1|
|              NULL|2025-01-01 02:36:46|        wolzsley32|            true|           3|             0|When are we going...|  ChatGPT|incre

## Save sampling and rest_data to Data Lake

In [ ]:
target_path_parquet = "gs://reddit-ai-2/process_data/Sampling_Post_for_LLM/sampling_post_for_LLM.parquet"

sampling.write.mode("overwrite").parquet(target_path_parquet)

print(f"บันทึกไฟล์ sampling_for_LLM.parquet เรียบร้อยแล้ว")

target_path_parquet = "gs://reddit-ai-2/process_data/Post_Single_or_Non_Tag/post_single_tag_or_non.parquet"

rest_data.write.mode("overwrite").parquet(target_path_parquet)

print(f"บันทึกไฟล์ post_single_tag_or_non.parquet เรียบร้อยแล้ว")

sampling.unpersist()
rest_data.unpersist()

26/04/26 09:36:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:36:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:36:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:36:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:36:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 09:36:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/26 0

บันทึกไฟล์ sampling_for_LLM.parquet เรียบร้อยแล้ว


บันทึกไฟล์ post_single_tag_or_non.parquet เรียบร้อยแล้ว


DataFrame[content_categories: string, created_utc: string, author: string, is_crosspostable: boolean, num_comments: bigint, num_crossposts: bigint, selftext: string, subreddit: string, title: string, upvote_ratio: double, ups: bigint, downs: bigint, view_count: string, AI_Name: string]

In [ ]:
spark.stop()